# Bài 10: Gọi API

## Tại sao cần học gọi API?

Ở Bài 9, agent của bạn đã dùng DuckDuckGo để tìm kiếm web. Nhưng bên dưới, nó hoạt động thế nào?

Thực chất, tool DuckDuckGo **gọi một API** — nó gửi yêu cầu qua internet và nhận dữ liệu về. Mọi tool mà agent dùng đều hoạt động theo cách này:
- Tool tìm kiếm gọi API tìm kiếm
- Tool hình ảnh gọi API hình ảnh
- Tool database gọi API database

Trong bài này, bạn sẽ học cách tự gọi API. Đây chính là cách bạn sẽ xây dựng **custom tool** cho agent sau này.

> **Chi phí:** Bài này dùng API miễn phí cho hầu hết ví dụ. Bài tập cuối dùng DataForSEO (không bắt buộc, cần API key). Nếu chưa có DataForSEO key, bạn vẫn theo được — các khái niệm giống nhau.

In [ ]:
# Cài httpx — thư viện HTTP hiện đại cho Python
!python -m pip install httpx python-dotenv

## API là gì?

**API** (Application Programming Interface) là cách các chương trình giao tiếp với nhau qua internet.

Hiểu đơn giản như đặt đồ ăn tại nhà hàng:
- **Bạn** (code) gửi **đơn hàng** (request) đến **bếp** (server)
- Bếp chuẩn bị món và gửi lại **phản hồi** (response)
- Bạn không cần biết bếp hoạt động thế nào — chỉ cần biết **thực đơn** (tài liệu API)

Khi gọi API, bạn:
1. Gửi một **request** đến URL (endpoint của API)
2. Nhận lại **response** chứa dữ liệu (thường ở định dạng JSON)

## HTTP cơ bản: GET và POST

Có hai loại request chính:

| Method | Khi nào dùng | Ví dụ |
|--------|------------|----------|
| **GET** | Lấy dữ liệu | "Cho tôi thời tiết Hà Nội" |
| **POST** | Gửi dữ liệu | "Đây là keyword, hãy phân tích" |

Mỗi request còn có:
- **URL** — Gửi đến đâu (ví dụ: `https://api.example.com/search`)
- **Headers** — Thông tin bổ sung như token xác thực
- **Body** — Dữ liệu gửi kèm (chỉ với POST)
- **Status code** — Response cho biết kết quả:
  - `200` = Thành công
  - `401` = Không được phép (API key sai)
  - `404` = Không tìm thấy
  - `429` = Quá nhiều request (bị giới hạn)
  - `500` = Lỗi server

## Gửi request với `httpx`

Python có nhiều thư viện cho HTTP request. Chúng ta dùng **httpx** vì nó hiện đại, nhanh, và đã có sẵn trong project.

Bắt đầu với một GET request đến API miễn phí.

In [ ]:
import httpx

# GET request đến API công khai miễn phí
response = httpx.get("https://httpbin.org/get")

print(f"Status code: {response.status_code}")
print(f"Content type: {response.headers['content-type']}")
print(f"Response body (300 ký tự đầu):")
print(response.text[:300])

In [ ]:
# POST request — gửi dữ liệu đến server
response = httpx.post(
    "https://httpbin.org/post",
    json={"keyword": "on-page SEO", "location": "United States"},
)

print(f"Status code: {response.status_code}")

# Response cho thấy server đã nhận dữ liệu gì
data = response.json()
print(f"Server nhận được: {data['json']}")

## Đọc JSON response

Hầu hết API trả về dữ liệu dạng **JSON** — tương ứng trực tiếp với dict và list trong Python.

Pattern luôn là:
```python
response = httpx.get("https://api.example.com/data")
data = response.json()  # Chuyển JSON string → Python dict
value = data["key"]     # Lấy giá trị cần thiết
```

Thử với một API thật trả về dữ liệu hữu ích.

In [ ]:
# Gọi API miễn phí trả về dữ liệu người dùng ngẫu nhiên
response = httpx.get("https://randomuser.me/api/?results=3")
data = response.json()

# Trích xuất thông tin cần thiết từ JSON lồng nhau
for user in data["results"]:
    name = f"{user['name']['first']} {user['name']['last']}"
    country = user["location"]["country"]
    email = user["email"]
    print(f"{name} từ {country} — {email}")

## API key và xác thực

Hầu hết API hữu ích đều yêu cầu **xác thực** (authentication) — cần biết ai đang gọi.

Các cách phổ biến:
1. **API key trong header** — Phổ biến nhất: `headers={"Authorization": "Bearer YOUR_KEY"}`
2. **Basic auth** — Tên đăng nhập + mật khẩu: `auth=("login", "password")`
3. **API key trong URL** — Ít bảo mật hơn: `?api_key=YOUR_KEY`

**Quan trọng:** Không bao giờ đặt API key trực tiếp trong code. Dùng biến môi trường (env):

```python
import os
api_key = os.getenv("MY_API_KEY")  # Đọc từ file .env
```

In [ ]:
# Ví dụ: API key trong header
response = httpx.get(
    "https://httpbin.org/headers",
    headers={"Authorization": "Bearer my-secret-key-123"},
)
print("Server thấy các headers này:")
print(response.json()["headers"])

print()

# Ví dụ: Basic auth (tên đăng nhập + mật khẩu)
response = httpx.get(
    "https://httpbin.org/basic-auth/myuser/mypass",
    auth=("myuser", "mypass"),
)
print(f"Kết quả Basic auth: {response.json()}")

## Xử lý lỗi

API có thể lỗi vì nhiều lý do: key sai, mạng có vấn đề, bị giới hạn tốc độ. Luôn xử lý lỗi:

```python
try:
    response = httpx.get(url, timeout=15)
    response.raise_for_status()  # Báo lỗi nếu status code 4xx/5xx
    data = response.json()
except httpx.HTTPStatusError as e:
    print(f"API trả lỗi: {e.response.status_code}")
except httpx.RequestError as e:
    print(f"Lỗi mạng: {e}")
```

Điểm chính:
- Luôn đặt **timeout** để code không treo mãi
- Dùng `raise_for_status()` để bắt response lỗi
- Xử lý từng loại lỗi riêng biệt

In [ ]:
# Xử lý lỗi đúng cách

def safe_api_call(url):
    """Gọi API với xử lý lỗi đầy đủ."""
    try:
        response = httpx.get(url, timeout=15)
        response.raise_for_status()
        return response.json()
    except httpx.HTTPStatusError as e:
        print(f"Lỗi API {e.response.status_code}: {e.response.text[:100]}")
        return None
    except httpx.RequestError as e:
        print(f"Lỗi mạng: {e}")
        return None

# URL đúng — sẽ thành công
print("URL đúng:")
result = safe_api_call("https://httpbin.org/get")
print(f"  Có dữ liệu: {bool(result)}")

# URL sai — sẽ trả 404
print("\nURL sai (404):")
result = safe_api_call("https://httpbin.org/status/404")
print(f"  Có dữ liệu: {bool(result)}")

## Tổng hợp: Gọi API thật

Kết hợp tất cả vào một hàm gọi API **DataForSEO** SERP để xem **AI Overview** của Google nói gì về một keyword.

Đây chính là cách sản phẩm hoạt động — tool phân tích AIO gọi cùng API này.

> **Lưu ý:** Cần `DATA_FOR_SEO_API_KEY` trong file `.env`. Nếu chưa có, hãy đọc code và output — các khái niệm giống nhau dù dùng API nào.

In [ ]:
import os
import base64
from dotenv import load_dotenv

load_dotenv()


def get_ai_overview(keyword):
    """Kiểm tra AI Overview của Google nói gì về một keyword."""
    # Bước 1: Lấy credentials từ biến môi trường
    raw_key = os.getenv("DATA_FOR_SEO_API_KEY", "")
    if not raw_key:
        print("DATA_FOR_SEO_API_KEY chưa được cài — bỏ qua API call.")
        return None

    # Bước 2: Giải mã Basic auth credentials
    if raw_key.startswith("Basic "):
        raw_key = raw_key[len("Basic "):]
    decoded = base64.b64decode(raw_key).decode("utf-8")
    login, password = decoded.split(":", 1)

    # Bước 3: Gửi POST request
    try:
        response = httpx.post(
            "https://api.dataforseo.com/v3/serp/google/organic/live/advanced",
            auth=(login, password),
            json=[{
                "keyword": keyword,
                "location_code": 2840,       # United States
                "language_code": "en",
                "load_async_ai_overview": True,
            }],
            timeout=60,
        )
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Gọi API thất bại: {e}")
        return None

    # Bước 4: Đọc response — tìm phần AI Overview
    tasks = data.get("tasks", [])
    if not tasks or not tasks[0].get("result"):
        return {"keyword": keyword, "has_aio": False}

    items = tasks[0]["result"][0].get("items", [])
    for item in items:
        if item.get("type") == "ai_overview":
            sections = []
            references = []
            for element in item.get("items", []):
                if element.get("text"):
                    sections.append(element["text"])
                for ref in element.get("references", []):
                    references.append(ref.get("url", ""))

            return {
                "keyword": keyword,
                "has_aio": True,
                "content": "\n\n".join(sections),
                "sources_cited": references,
            }

    return {"keyword": keyword, "has_aio": False}


# Thử luôn!
result = get_ai_overview("on-page SEO best practices")
if result:
    print(f"Keyword: {result['keyword']}")
    print(f"Có AI Overview: {result['has_aio']}")
    if result['has_aio']:
        print(f"\nNội dung AI Overview (500 ký tự đầu):")
        print(result['content'][:500])
        print(f"\nNguồn được trích dẫn: {len(result['sources_cited'])}")
        for url in result['sources_cited'][:5]:
            print(f"  - {url}")

## Kết nối với agent

Mọi thứ bạn vừa học chính là những gì xảy ra bên trong tool của agent:

1. **DuckDuckGoTools** gọi API DuckDuckGo (GET request)
2. **FreepikTools** gọi API Freepik (GET với API key header)
3. **AIOTools** gọi API DataForSEO (POST với Basic auth)

Khi bạn xây custom tool cho agent, bạn đang viết một hàm:
- Nhận tham số từ agent
- Gọi API bằng `httpx`
- Trả về kết quả đã xử lý

Trong bài tiếp theo, bạn sẽ học cách làm agent trả về **dữ liệu có cấu trúc** — các trường cụ thể trong format dự đoán được. Sau đó, bạn sẽ nối các agent lại thành pipeline.

## Tóm tắt bài 10

Bạn đã học:
- **API** là cách các chương trình giao tiếp qua internet
- **GET** lấy dữ liệu, **POST** gửi dữ liệu
- **httpx** gửi HTTP request trong Python
- **API key** lưu trong biến môi trường, không bao giờ đặt trong code
- **Luôn xử lý lỗi** — đặt timeout, kiểm tra status code
- **Đọc JSON** — `response.json()` cho bạn Python dict
- Tool của agent chính là các hàm gọi API

**Bài tiếp theo:** Structured output — làm agent trả dữ liệu theo format cụ thể.

## Bài tập

Viết hàm gọi API [JSONPlaceholder](https://jsonplaceholder.typicode.com) để:
1. Lấy danh sách bài viết từ `https://jsonplaceholder.typicode.com/posts`
2. Lọc chỉ lấy bài viết của user ID 1
3. In tiêu đề mỗi bài

Nhớ xử lý lỗi với try/except.

In [ ]:
# Bài tập: Điền vào chỗ trống

import httpx


def get_user_posts(user_id):
    """Lấy bài viết của một user từ JSONPlaceholder."""
    try:
        response = httpx.get(
            ___,          # URL API
            timeout=___,  # Đặt timeout
        )
        response.raise_for_status()
        all_posts = response.json()

        # Lọc bài viết theo user_id
        user_posts = [p for p in all_posts if p[___] == user_id]
        return user_posts

    except Exception as e:
        print(f"Lỗi: {e}")
        return []


posts = get_user_posts(1)
print(f"Tìm thấy {len(posts)} bài viết của user 1:\n")
for post in posts:
    print(f"  - {post['title']}")

<details>
<summary>Bấm để xem đáp án</summary>

```python
import httpx


def get_user_posts(user_id):
    """Lấy bài viết của một user từ JSONPlaceholder."""
    try:
        response = httpx.get(
            "https://jsonplaceholder.typicode.com/posts",
            timeout=15,
        )
        response.raise_for_status()
        all_posts = response.json()

        user_posts = [p for p in all_posts if p["userId"] == user_id]
        return user_posts

    except Exception as e:
        print(f"Lỗi: {e}")
        return []


posts = get_user_posts(1)
print(f"Tìm thấy {len(posts)} bài viết của user 1:\n")
for post in posts:
    print(f"  - {post['title']}")
```
</details>